# 05 · 窗口函数（面试最高频，本章最厚）

> 这是整个 SQL 模块的“皇冠”。**窗口函数 = 在不折叠行的前提下，做聚合 / 排名 / 取前后行。**
> 你在 Pandas 里其实天天在用：`df.rank()`、`df['sales'].cumsum()`、`df.groupby('cat')['sales'].shift()`、滚动平均 `rolling(3).mean()`——这些在 SQL 里全是窗口函数。面试官爱考它，是因为它正好覆盖“组内 TopN、环比同比、累计、去重、异常检测”这些真实业务问题。
>
> 学完这一课，你会觉得 04 课那个“手写组内 TopN”又臭又长的 SQL 简直是黑历史。

## 本课地图
1. 核心思想：窗口函数不折叠行（和 GROUP BY 的本质区别）
2. 语法骨架：`函数() OVER (PARTITION BY ... ORDER BY ... frame)`
3. 排序三兄弟：ROW_NUMBER / RANK / DENSE_RANK（组内 TopN、去重）
4. 偏移两兄弟：LAG / LEAD（环比、同比、算间隔）
5. 聚合当窗口：SUM/AVG OVER（累计值、移动平均，frame 详解）
6. FIRST_VALUE / NTILE（取组内第一、分桶）
7. MySQL 8 / PostgreSQL 方言对照

In [1]:
# 统一环境：连接数据库（绝对路径，可反复运行）
import sqlite3
import pandas as pd

# 两个库都用绝对路径写死，避免“相对路径在哪个工作目录下”的坑
SHOP_DB = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\shop.db"
SS_DB  = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\superstore.db"

# 让 DataFrame 打印好看一点（你已经熟悉 Pandas，这里直接拿它当 SQL 的“显示层”）
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# 教学默认连 shop.db；需要 superstore 时再切
conn = sqlite3.connect(SHOP_DB)

def q(sql, conn=conn):
    """执行一条 SQL，把结果表直接返回成 Pandas DataFrame。"""
    return pd.read_sql_query(sql, conn)

print("已连接 shop.db。可用 q('SELECT ...') 跑查询。")
print("shop.db 表：", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

已连接 shop.db。可用 q('SELECT ...') 跑查询。
shop.db 表： ['customers', 'employees', 'order_items', 'orders', 'products']


## 1. 核心思想：窗口函数“不折叠”行

先回顾 `GROUP BY`：它把 6407 条明细按类别压成 3 行，每组只剩一行——**细节行丢了**。

窗口函数不一样：它**一行都不删**，只是在每一行旁边，附上“这一行所在窗口的统计值”。

In [2]:
# 对比：左边 GROUP BY 折叠，右边窗口函数不折叠
# 窗口版：每条明细都在，同时多出一列“该类别总销售额”
q('''
SELECT p.category, p.product_name, oi.sales,
       SUM(oi.sales) OVER (PARTITION BY p.category) AS 类别总额,
       ROUND( oi.sales * 100.0 / SUM(oi.sales) OVER (PARTITION BY p.category), 2 ) AS 占类别百分比
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
ORDER BY p.category, oi.sales DESC
LIMIT 10
''')

,category,product_name,sales,类别总额,占类别百分比
0,办公用品,装订机-02号,1098.36,396468.46,0.28
1,办公用品,装订机-02号,1098.36,396468.46,0.28
2,办公用品,装订机-02号,1098.36,396468.46,0.28
3,办公用品,装订机-02号,1098.36,396468.46,0.28
4,办公用品,装订机-02号,1098.36,396468.46,0.28
5,办公用品,装订机-02号,1098.36,396468.46,0.28
6,办公用品,装订机-02号,1098.36,396468.46,0.28
7,办公用品,装订机-02号,1098.36,396468.46,0.28
8,办公用品,装订机-02号,1098.36,396468.46,0.28
9,办公用品,装订机-02号,1098.36,396468.46,0.28


看出来了吗？`SUM(oi.sales) OVER (PARTITION BY p.category)` 没有把多行压成一行——它算出“该类别的总额”，然后**广播到这个类别里的每一行**。这就是窗口函数的灵魂：**每行都保留，同时带一个组级统计**。

> 类比 Pandas：`df.groupby('category')['sales'].transform('sum')`——transform 就是“组聚合后回填到每行”，和窗口函数一模一样。

## 2. 语法骨架

```
函数名(参数) OVER (
    PARTITION BY  分组列      -- 像 groupby，把行切成多个小窗口（可省略=全体一个窗口）
    ORDER BY      排序列      -- 窗口内行的排序（排名/累计/偏移函数需要）
    ROWS BETWEEN ...          -- 帧(frame)：当前行往前后取几行（聚合窗口用）
)
```

记成一句话：**OVER 定义“窗口长啥样”，前面的函数定义“在这个窗口上干啥”。**

## 3. 排序三兄弟：ROW_NUMBER / RANK / DENSE_RANK

这是最常用的一组，全靠 `OVER (PARTITION BY ... ORDER BY ...)`。

| 函数 | 并列时怎么排 | 举例（分数 100,100,90） |
|---|---|---|
| `ROW_NUMBER()` | 不允许并列，硬编唯一序号 | 1,2,3 |
| `RANK()` | 并列同名次，**之后跳号** | 1,1,3 |
| `DENSE_RANK()` | 并列同名次，**之后不跳号** | 1,1,2 |

In [3]:
# 组内 TopN：每个销售额最高的商品，在它自己类别里排第几？
# 先按类别+商品聚合出每个商品的总销售额，再在类别内排名
q('''
WITH prod_sales AS (
    SELECT p.category, p.product_name, SUM(oi.sales) AS total_sales
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.category, p.product_name
)
SELECT category, product_name, total_sales,
       ROW_NUMBER() OVER (PARTITION BY category ORDER BY total_sales DESC) AS rn,
       RANK()       OVER (PARTITION BY category ORDER BY total_sales DESC) AS rnk,
       DENSE_RANK() OVER (PARTITION BY category ORDER BY total_sales DESC) AS drnk
FROM prod_sales
ORDER BY category, rn
LIMIT 12
''')

,category,product_name,total_sales,rn,rnk,drnk
0,办公用品,装订机-02号,59457.82,1,1,1
1,办公用品,装订机-01号,48376.55,2,2,2
2,办公用品,装订机-04号,48204.00,3,3,3
3,办公用品,装订机-03号,36355.28,4,4,4
4,办公用品,收纳-03号,24840.55,5,5,5
5,办公用品,收纳-04号,20129.80,6,6,6
6,办公用品,收纳-01号,18874.24,7,7,7
7,办公用品,纸张-02号,18572.03,8,8,8
8,办公用品,纸张-04号,16583.48,9,9,9
9,办公用品,收纳-02号,15111.58,10,10,10


In [4]:
# 实战：每个类别销售额 Top2 的商品（组内 TopN 的标准写法）
# 套一层 WHERE rn <= 2 即可——注意窗口函数不能直接写在 WHERE 里
q('''
WITH prod_sales AS (
    SELECT p.category, p.product_name, SUM(oi.sales) AS total_sales
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.category, p.product_name
),
ranked AS (
    SELECT category, product_name, total_sales,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY total_sales DESC) AS rn
    FROM prod_sales
)
SELECT * FROM ranked WHERE rn <= 2
ORDER BY category, rn
''')

,category,product_name,total_sales,rn
0,办公用品,装订机-02号,59457.82,1
1,办公用品,装订机-01号,48376.55,2
2,家具,沙发-03号,1051185.09,1
3,家具,沙发-02号,958645.94,2
4,科技产品,电脑-04号,2274612.73,1
5,科技产品,电脑-03号,1738469.10,2


> 关键约束：**窗口函数在 SELECT 阶段才算，不能写进 WHERE / ON / GROUP BY**。所以“按排名筛选”必须套一层子查询或 CTE（这里用了 `WITH`，06 课细讲），把排名先算出来，外层再 `WHERE rn <= 2`。这也是为什么 04 课要先讲子查询。

### 用 ROW_NUMBER 去重

窗口函数另一个杀手锏：**每组只保留一行**（比如每个客户只保留最新一单）。

In [5]:
# 每个客户最新的一单：按客户分组、下单日期倒序编号，取 rn=1
q('''
WITH ordered AS (
    SELECT o.order_id, o.customer_id, o.order_date,
           ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date DESC) AS rn
    FROM orders o
)
SELECT order_id, customer_id, order_date
FROM ordered
WHERE rn = 1
ORDER BY customer_id
LIMIT 10
''')

,order_id,customer_id,order_date
0,452,1,2025-12-14
1,1859,2,2025-06-25
2,182,3,2025-12-06
3,875,4,2025-10-28
4,659,5,2025-11-23
5,1002,6,2025-12-04
6,1177,7,2025-12-04
7,1917,8,2025-08-03
8,1962,9,2025-08-03
9,571,10,2025-11-18


> 这比“groupby 取 max 再 join”干净太多。要注意：如果并列日期，ROW_NUMBER 会随机挑一个，需要再加一个 tie-breaker（比如 `ORDER BY order_date DESC, order_id DESC`）让结果确定。

## 4. 偏移两兄弟：LAG / LEAD

`LAG(列, n)` 取“当前行往上数第 n 行”的值；`LEAD` 反之取往下。它们是**环比、同比、算间隔**的核心。

In [6]:
# 每月销售额，以及和上一个月的差（环比增量）
q('''
WITH monthly AS (
    SELECT strftime('%Y-%m', order_date) AS ym, SUM(oi.sales) AS month_sales
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY ym
)
SELECT ym,
       month_sales,
       LAG(month_sales, 1) OVER (ORDER BY ym) AS 上月销售额,
       ROUND( (month_sales - LAG(month_sales,1) OVER (ORDER BY ym))
              / LAG(month_sales,1) OVER (ORDER BY ym) * 100, 1) AS 环比_pct
FROM monthly
ORDER BY ym
LIMIT 12
''')

,ym,month_sales,上月销售额,环比_pct
0,2022-01,420034.19,NaN,NaN
1,2022-02,451637.56,420034.19,7.5
2,2022-03,613911.95,451637.56,35.9
3,2022-04,472740.49,613911.95,-23.0
4,2022-05,500490.98,472740.49,5.9
5,2022-06,569560.00,500490.98,13.8
6,2022-07,630151.31,569560.00,10.6
7,2022-08,442876.74,630151.31,-29.7
8,2022-09,414188.51,442876.74,-6.5
9,2022-10,494728.91,414188.51,19.4


> `LAG(x,1)` 让“本行”和“上一行”在同一行里可比——这正是 Pandas `df['sales'].shift(1)`。第一行没有上一行，结果是 NULL，做除法时要小心 NULL。
>
> **同比**（和去年同月比）就是 `LAG(month_sales, 12)`；**环比**是 `LAG(...,1)`。换个数字就是。

## 5. 聚合当窗口：累计值、移动平均（frame 详解）

把聚合函数放进 `OVER`，就得到“累计 / 滑动”统计。这里要引入**帧 frame**：窗口内到底“往前后包几行”。

In [7]:
# 累计销售额：从年初累加到当前月
# frame 默认值（有 ORDER BY 时）就是 RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
q('''
WITH monthly AS (
    SELECT strftime('%Y-%m', order_date) AS ym, SUM(oi.sales) AS month_sales
    FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY ym
)
SELECT ym, month_sales,
       SUM(month_sales) OVER (ORDER BY ym
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS 累计销售额
FROM monthly
ORDER BY ym
LIMIT 12
''')

,ym,month_sales,累计销售额
0,2022-01,420034.19,420034.19
1,2022-02,451637.56,871671.75
2,2022-03,613911.95,1485583.70
3,2022-04,472740.49,1958324.19
4,2022-05,500490.98,2458815.17
5,2022-06,569560.00,3028375.17
6,2022-07,630151.31,3658526.48
7,2022-08,442876.74,4101403.22
8,2022-09,414188.51,4515591.73
9,2022-10,494728.91,5010320.64


帧的三种写法，背下来：

```
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW     -- 从第一行到当前行（累计）
ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING             -- 前一行 + 当前 + 后一行（3 期移动）
ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING  -- 整个窗口（=不带 ORDER 的分组聚合）
```

In [8]:
# 移动平均：当月 + 前两月，共 3 期平均，平滑月度波动
q('''
WITH monthly AS (
    SELECT strftime('%Y-%m', order_date) AS ym, SUM(oi.sales) AS month_sales
    FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY ym
)
SELECT ym, month_sales,
       ROUND(AVG(month_sales) OVER (ORDER BY ym
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS 三期移动平均
FROM monthly
ORDER BY ym
LIMIT 12
''')

,ym,month_sales,三期移动平均
0,2022-01,420034.19,420034.2
1,2022-02,451637.56,435835.9
2,2022-03,613911.95,495194.6
3,2022-04,472740.49,512763.3
4,2022-05,500490.98,529047.8
5,2022-06,569560.00,514263.8
6,2022-07,630151.31,566734.1
7,2022-08,442876.74,547529.3
8,2022-09,414188.51,495738.9
9,2022-10,494728.91,450598.1


> 易错点：**`SUM(x) OVER (...)` 不写 ORDER BY，就是“整个分区求和”；一旦写了 ORDER BY，默认帧变成“从分区首行到当前行”，语义立刻变成累计求和。** 这个区别是无数 bug 的根源。

## 6. FIRST_VALUE 与 NTILE

- `FIRST_VALUE(列)`：取窗口内排序后第一行的值（常和累计/分组一起用）。
- `NTILE(n)`：把窗口内行尽量均匀地分成 n 组，返回 1~n 的组号。分桶利器。

In [9]:
# NTILE：把 60 个商品按总销售额分成 4 档（白金/黄金/白银/青铜）
q('''
WITH prod AS (
    SELECT p.product_name, SUM(oi.sales) AS total_sales
    FROM order_items oi JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.product_name
)
SELECT product_name, total_sales,
       NTILE(4) OVER (ORDER BY total_sales DESC) AS 销售档位
FROM prod
ORDER BY 销售档位, total_sales DESC
LIMIT 12
''')

,product_name,total_sales,销售档位
0,电脑-04号,2274612.73,1
1,电脑-03号,1738469.10,1
2,电脑-02号,1696164.58,1
3,手机-01号,1663743.56,1
4,电脑-01号,1248267.76,1
5,手机-02号,1127554.58,1
6,手机-04号,1095815.91,1
7,沙发-03号,1051185.09,1
8,平板-01号,970906.29,1
9,平板-03号,967238.48,1


In [10]:
# FIRST_VALUE：每个客户第一笔订单是哪天（和 LAG/LEAD 配合做 RFM 分析的零件）
q('''
SELECT customer_id, order_id, order_date,
       FIRST_VALUE(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS 首单日期,
       order_date AS 当前订单
FROM orders
WHERE customer_id IN (1,2,3)
ORDER BY customer_id, order_date
LIMIT 12
''')

,customer_id,order_id,order_date,首单日期,当前订单
0,1,1584,2022-01-08,2022-01-08,2022-01-08
1,1,641,2022-06-06,2022-01-08,2022-06-06
2,1,684,2022-10-03,2022-01-08,2022-10-03
3,1,2110,2022-10-06,2022-01-08,2022-10-06
4,1,212,2023-07-02,2022-01-08,2023-07-02
5,1,1439,2023-07-15,2022-01-08,2023-07-15
6,1,2563,2023-12-15,2022-01-08,2023-12-15
7,1,2064,2024-09-27,2022-01-08,2024-09-27
8,1,82,2025-04-02,2022-01-08,2025-04-02
9,1,2261,2025-07-18,2022-01-08,2025-07-18


## 7. MySQL 8 / PostgreSQL 方言对照

窗口函数在主流库里语法几乎一致，但历史包袱要知道：

| 数据库 | 窗口函数支持 | 备注 |
|---|---|---|
| **SQLite 3.25+** | 完整支持（你的 3.53.4 没问题） | 本课程已全程跑通 |
| **PostgreSQL** | 支持最完整，文档最好 | 工程师首选 |
| **MySQL 8.0+** | 完整支持 | **MySQL 5.x 不支持窗口函数！** 老面试题会问你 5.x 怎么绕 |
| SQLServer / Oracle | 支持（Oracle 叫分析函数） | 语法相同 |

> 面试若被问“MySQL 5.7 没有窗口函数怎么办”：那就是 04 课手写相关子查询 / 变量模拟排名——但现实中 5.7 已淘汰，知道即可，不用练。

## 动手小练习（项目里会用到）

1. 每个地区 `region` 销售额 Top3 的城市（组内 TopN）。
2. 2022~2025 每年的总销售额，并用 LAG 算同比增长。
3. 把所有客户按累计消费额分成 5 档（NTILE），看每档人数。

## 本课小结

- 窗口函数不折叠行：每行保留，旁边附一个“组级/窗口级”统计。
- `函数() OVER (PARTITION BY 分组 ORDER BY 排序 frame)`。
- ROW_NUMBER/RANK/DENSE_RANK 处理并列不同；组内 TopN 靠它 + 外层 WHERE rn<=N。
- LAG/LEAD 算环比同比、间隔；SUM/AVG OVER + frame 算累计、移动平均。
- 窗口函数在 SELECT 阶段算，不能进 WHERE，要套 CTE/子查询。
- 写 ORDER BY 后默认帧是“首行到当前行”，这是累计求和的来源，也是易错点。

## 来源与延伸

- SQLite 官方窗口函数文档（权威，含 frame 细则）：https://www.sqlite.org/windowfunctions.html
- Mode SQL Tutorial · 窗口函数（讲解清晰）：https://mode.com/sql-tutorial/sql-window-functions/
- SQLBolt 无专门章节，可配合 SQLZoo 高级练习：https://sqlzoo.net/
- PostgreSQL Tutorial · Window Functions：https://www.postgresqltutorial.com/postgresql-window-function/
- LeetCode Database（窗口函数题是 Medium/Hard 主力，如“部门工资前三高的所有员工”）：https://leetcode.com/problemset/database/
- DataLemur（面试真题，窗口函数占比极高）：https://datalemur.com/